# 06 -- Metadata & Advanced Filtering

## What We'll Cover
1. Adding custom metadata to memories
2. AND/OR compound filters
3. Filtering by metadata fields
4. Container-based organization strategies
5. Filtered writes (context-aware ingestion)


In [ ]:
# 6.1 Adding memories with rich metadata
from supermemory import Supermemory
client = Supermemory()

# Metadata can be any key-value pairs (strings, numbers, booleans)
docs = [
    {
        "content": "Q4 revenue increased 23% YoY. Cloud services drove 60% of growth.",
        "container_tag": "company_reports",
        "metadata": {"category": "financial", "quarter": "Q4", "year": 2025, "priority": "high"}
    },
    {
        "content": "New authentication service to launch in Q3. Supports OAuth2 and SAML.",
        "container_tag": "company_reports",
        "metadata": {"category": "engineering", "status": "planned", "team": "platform"}
    },
    {
        "content": "Hiring: 3 senior engineers for the ML platform team. Remote-friendly.",
        "container_tag": "company_reports",
        "metadata": {"category": "hr", "status": "open", "team": "ml-platform"}
    },
    {
        "content": "Customer satisfaction score reached 94% in Q4, up from 91%.",
        "container_tag": "company_reports",
        "metadata": {"category": "financial", "quarter": "Q4", "year": 2025}
    },
]

for doc in docs:
    r = client.add(**doc)
    print(f"Added [{doc['metadata']['category']}]: {r.id}")

print(f"\nTotal: {len(docs)} documents with metadata")


In [ ]:
# 6.2 Search with metadata filters -- AND condition
# Find financial documents from Q4 2025
results = client.search.documents(
    q="revenue growth",
    container_tags=["company_reports"],
    filters={
        "AND": [
            {"key": "category", "value": "financial"},
            {"key": "year", "value": 2025}
        ]
    }
)

print(f"Found {len(results.results)} financial documents from 2025:")
for r in results.results:
    print(f"  - {r.get('memory', str(r))[:100]}...")


In [ ]:
# 6.3 Search with OR filter
# Find documents about engineering OR hr
results = client.search.documents(
    q="team updates",
    container_tags=["company_reports"],
    filters={
        "OR": [
            {"key": "category", "value": "engineering"},
            {"key": "category", "value": "hr"}
        ]
    }
)

print(f"Found {len(results.results)} engineering or HR documents:")
for r in results.results:
    print(f"  - {r.get('memory', str(r))[:100]}...")


In [ ]:
# 6.4 Combined AND/OR filters
results = client.search.documents(
    q="",  # Empty query = return all matching filters
    container_tags=["company_reports"],
    filters={
        "AND": [
            {"OR": [
                {"key": "category", "value": "financial"},
                {"key": "category", "value": "engineering"}
            ]},
            {"key": "status", "value": "open"}
        ]
    }
)
print(f"Results: {len(results.results)}")
print("Filter: (category=financial OR category=engineering) AND status=open")
for r in results.results:
    print(f"  - {r.get('memory', str(r))[:100]}...")


In [ ]:
# 6.5 Container-based organization strategies
print("Container Tag Strategy Patterns:")
print()
print("1. USER-SCOPED: container_tag = 'user_<id>'")
print("   Best for: chatbots, personal assistants, per-user profiles")
print()
print("2. PROJECT-SCOPED: container_tag = 'project_<name>'")
print("   Best for: team projects, codebases, documentation")
print()
print("3. MULTI-DIMENSIONAL: containerTags = ['user_X', 'project_Y', 'team_Z']")
print("   Best for: cross-cutting concerns, matrix organizations")
print()
print("4. TENANT-SCOPED: Use userId for strict data partitioning")
print("   Best for: multi-tenant SaaS, isolated customer data")


In [ ]:
# 6.6 Filtered writes -- context-aware ingestion
# filterByMetadata: only use memories matching this filter as context during ingestion
# This ensures new memories connect to relevant existing context

print("Filtered Write Example:")
print("When adding new content about the ML platform,")
print("only use existing memories tagged with team='ml-platform' as context.")
print()
print("client.add(")
print("    content='New ML model deployment pipeline is ready',")
print("    container_tag='company_reports',")
print("    filter_by_metadata={'AND': [{'key': 'team', 'value': 'ml-platform'}]}")
print(")")
print()
print("This ensures the new memory connects to relevant existing context.")


## 6.7 Best Practices for Metadata

- **Define a schema early** -- retrofitting metadata is hard
- **Use consistent keys** -- `category`, `status`, `team`, `priority` are good foundations
- **Keep values simple** -- strings, numbers, booleans
- **Combine tags + metadata** -- tags for scoping, metadata for filtering
- **Test filters** -- make sure your filter queries return expected results

## 6.8 Key Takeaways

- Metadata enables powerful filtering (AND, OR, nested combinations)
- Container tags provide the primary scoping mechanism
- Use `containerTags` for multi-dimensional organization
- Filters work with search, list, and filtered writes
- Organize early -- it's hard to retroactively add metadata

**Next:** Notebook 07 -- Async Operations
